<a href="https://colab.research.google.com/github/vishaanthg08-source/TrendPulse/blob/part1-cleaning/part1_python_foundations/part1_python_foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TrendPulse - Part 1: Python Foundations
Core Python on YouTube trending data (US, GB, IN).


In [33]:
!git clone -q -b part1-cleaning https://github.com/vishaanthg08-source/trendpulse.git
%cd trendpulse/part1_python_foundations

/content/trendpulse/part1_python_foundations/trendpulse/part1_python_foundations/trendpulse/part1_python_foundations/trendpulse/part1_python_foundations


In [34]:
import os
print(os.listdir("data"))

['.gitkeep', 'USvideos.csv', 'GBvideos.csv', 'categories.csv', 'INvideos.csv']


In [35]:
from datetime import datetime
import pandas as pd

# ---- Configuration ----
DATA_DIR = "data"                    # str
COUNTRIES = ["US", "GB", "IN"]       # list of str
CATEGORIES_FILE = f"{DATA_DIR}/categories.csv"   # str
VIRAL_VIEWS = 1_000_000              # int: views for Viral Hit
GOOD_LIKE_RATIO = 0.90               # float: good like ratio
DIVISIVE_LIKE_RATIO = 0.60           # float: below this = Divisive
TOP_N_CHANNELS = 15                  # int
RUN_START = datetime.now()           # datetime

# ---- Run header ----
print("=" * 45)
print("TrendPulse - Part 1 run")
print(f"Countries : {', '.join(COUNTRIES)}")
print(f"Started   : {RUN_START:%Y-%m-%d %H:%M:%S}")
print("=" * 45)

TrendPulse - Part 1 run
Countries : US, GB, IN
Started   : 2026-10-30 17:20:14


In [36]:
REQUIRED_COLS = ["video_id", "trending_date", "title", "channel_title", "category_id",
                 "publish_time", "views", "likes", "dislikes", "comment_count",
                 "comments_disabled", "ratings_disabled", "video_error_or_removed"]

def load_country(path, country, max_rows=None):
    df = pd.read_csv(path, nrows=max_rows)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{country}: missing columns {missing}")
    df["country"] = country
    return df.to_dict("records")

In [37]:
all_records = []
for c in COUNTRIES:
    rows = load_country(f"{DATA_DIR}/{c}videos.csv", c)
    print(f"{c}: {len(rows)} records loaded")
    all_records.extend(rows)

print("Total records:", len(all_records))

US: 40949 records loaded
GB: 38916 records loaded
IN: 37352 records loaded
Total records: 117217


In [38]:
cat_rows = pd.read_csv(CATEGORIES_FILE).to_dict("records")
category_names = {row["category_id"]: row["category_name"] for row in cat_rows}
print(len(category_names), "categories")
print(category_names)

32 categories
{1: 'Film & Animation', 2: 'Autos & Vehicles', 10: 'Music', 15: 'Pets & Animals', 17: 'Sports', 18: 'Short Movies', 19: 'Travel & Events', 20: 'Gaming', 21: 'Videoblogging', 22: 'People & Blogs', 23: 'Comedy', 24: 'Entertainment', 25: 'News & Politics', 26: 'Howto & Style', 27: 'Education', 28: 'Science & Technology', 29: 'Nonprofits & Activism', 30: 'Movies', 31: 'Anime/Animation', 32: 'Action/Adventure', 33: 'Classics', 34: 'Comedy', 35: 'Documentary', 36: 'Drama', 37: 'Family', 38: 'Foreign', 39: 'Horror', 40: 'Sci-Fi/Fantasy', 41: 'Thriller', 42: 'Shorts', 43: 'Shows', 44: 'Trailers'}


In [39]:
def to_int(value):
    return int(float(value))

def to_bool(value):
    text = str(value).strip().lower()
    if text in ("true", "1"):
        return True
    if text in ("false", "0"):
        return False
    raise ValueError(f"not a bool: {value!r}")

def parse_trending_date(text):
    return datetime.strptime(str(text).strip(), "%y.%d.%m").date()

def parse_publish_time(text):
    return datetime.strptime(str(text).strip(), "%Y-%m-%dT%H:%M:%S.%fZ")

In [40]:
INT_FIELDS = ["views", "likes", "dislikes", "comment_count"]
BOOL_FIELDS = ["comments_disabled", "ratings_disabled", "video_error_or_removed"]

failures = {}
for name in INT_FIELDS + BOOL_FIELDS + ["trending_date", "publish_time"]:
    failures[name] = 0

clean_records = []
dropped = 0

In [41]:
for rec in all_records:
    new = dict(rec)
    bad = False

    for f in INT_FIELDS:
        try:
            new[f] = to_int(rec[f])
        except (ValueError, TypeError):
            failures[f] += 1
            bad = True

    for f in BOOL_FIELDS:
        try:
            new[f] = to_bool(rec[f])
        except ValueError:
            failures[f] += 1
            new[f] = False

    try:
        new["trending_date"] = parse_trending_date(rec["trending_date"])
    except ValueError:
        failures["trending_date"] += 1
        bad = True

    try:
        new["publish_time"] = parse_publish_time(rec["publish_time"])
    except ValueError:
        failures["publish_time"] += 1
        bad = True

    if bad:
        dropped += 1
    else:
        clean_records.append(new)

In [42]:
print("Before :", len(all_records))
print("After  :", len(clean_records))
print("Dropped:", dropped)
print("Failures per field:")
for name in failures:
    print(name, "=", failures[name])

Before : 117217
After  : 117217
Dropped: 0
Failures per field:
views = 0
likes = 0
dislikes = 0
comment_count = 0
comments_disabled = 0
ratings_disabled = 0
video_error_or_removed = 0
trending_date = 0
publish_time = 0


## Cleaning decisions
- views, likes, dislikes, comment_count: converted to int. A bad value drops the record, because replacing it with 0 would distort averages.
- True/False flags: converted to bool. A bad value is replaced with False.
- trending_date is stored as yy.dd.mm (17.14.11 = 14 Nov 2017), parsed with datetime.strptime.
- publish_time is ISO format, parsed with datetime.strptime.
- try/except wraps every conversion, so one bad value cannot crash the run.

All 117,217 records parsed successfully, so no record was dropped and every failure count is 0. The try/except handling is still in place for messy data.

In [49]:
key_tuples = []
for rec in clean_records:
    t = (rec["video_id"], rec["trending_date"], rec["views"], rec["likes"])
    key_tuples.append(t)

print(len(key_tuples))
print(key_tuples[0])

117217
('2kyS6SvSYSE', datetime.date(2017, 11, 14), 748374, 57527)


In [50]:
ids_by_country = {}
for c in COUNTRIES:
    ids = set()
    for rec in clean_records:
        if rec["country"] == c:
            ids.add(rec["video_id"])
    ids_by_country[c] = ids
    print(c, "unique videos:", len(ids))

US unique videos: 6351
GB unique videos: 3272
IN unique videos: 16307


In [51]:
us = ids_by_country["US"]
gb = ids_by_country["GB"]
india = ids_by_country["IN"]

in_all_three = us & gb & india
only_us = us - gb - india

print("Trended in all 3 countries:", len(in_all_three))
print("Only in US:", len(only_us))

Trended in all 3 countries: 79
Only in US: 5183


In [52]:
channel_counts = {}
for rec in clean_records:
    name = rec["channel_title"]
    if name in channel_counts:
        channel_counts[name] += 1
    else:
        channel_counts[name] = 1

print("Total channels:", len(channel_counts))

Total channels: 4452


In [53]:
sorted_channels = sorted(channel_counts.items(), key=lambda x: x[1], reverse=True)
top_15 = sorted_channels[:15]

for name, count in top_15:
    print(f"{name}: {count}")


WWE: 483
The Late Show with Stephen Colbert: 467
TheEllenShow: 453
The Tonight Show Starring Jimmy Fallon: 433
Jimmy Kimmel Live: 432
Netflix: 423
Late Night with Seth Meyers: 402
Saturday Night Live: 381
The Late Late Show with James Corden: 377
Warner Bros. Pictures: 355
Marvel Entertainment: 338
ESPN: 334
Screen Junkies: 323
20th Century Fox: 311
WIRED: 305


In [54]:
nested = {}
for rec in clean_records:
    country = rec["country"]
    cat = category_names.get(rec["category_id"], "Unknown")
    if country not in nested:
        nested[country] = {}
    if cat not in nested[country]:
        nested[country][cat] = 0
    nested[country][cat] += 1

for country in nested:
    print(country)
    for cat in nested[country]:
        print(f"  {cat}: {nested[country][cat]}")

US
  People & Blogs: 3210
  Entertainment: 9964
  Comedy: 3457
  Science & Technology: 2401
  Film & Animation: 2345
  News & Politics: 2487
  Sports: 2174
  Music: 6472
  Pets & Animals: 920
  Education: 1656
  Howto & Style: 4146
  Autos & Vehicles: 384
  Travel & Events: 402
  Gaming: 817
  Nonprofits & Activism: 57
  Shows: 57
GB
  Howto & Style: 1928
  Entertainment: 9124
  Music: 13754
  Sports: 1907
  News & Politics: 1225
  People & Blogs: 2926
  Comedy: 1828
  Science & Technology: 518
  Pets & Animals: 534
  Education: 457
  Film & Animation: 2577
  Gaming: 1788
  Autos & Vehicles: 144
  Travel & Events: 96
  Nonprofits & Activism: 90
  Shows: 20
IN
  Film & Animation: 1658
  News & Politics: 5241
  Entertainment: 16712
  Comedy: 3429
  Music: 3858
  People & Blogs: 2624
  Travel & Events: 8
  Science & Technology: 552
  Education: 1227
  Howto & Style: 845
  Shows: 205
  Sports: 731
  Nonprofits & Activism: 105
  Autos & Vehicles: 72
  Pets & Animals: 3
  Gaming: 66
  Movies

## Nested dictionary: country -> category -> count
- The outer dictionary has country names as keys.
- Each value is an inner dictionary: category name -> number of records.
- If a category ID is missing, "Unknown" is used, so the code does not crash.

## Why each data structure
- List of dictionaries (clean_records): stores all records, each record has named fields. Mutable.
- Tuple (key_tuples): holds 4 key fields that should not change. Immutable.
- Set (unique video IDs): removes duplicates automatically and lets me find common videos. Mutable.
- Dictionary counter (channel_counts): channel name is the key, count is the value. Mutable.
- Nested dictionary: data has two levels, country then category. Mutable.
- Slicing ([:15]): takes only the top 15 channels from the sorted list.

In [55]:
outside_var = "I am outside"

def scope_demo():
    inside_var = "I am inside"
    print("Inside function, can see outside:", outside_var)
    print("Inside function, can see inside:", inside_var)

scope_demo()
print("Outside, can see outside:", outside_var)

try:
    print(inside_var)
except NameError as e:
    print("Error:", e)

Inside function, can see outside: I am outside
Inside function, can see inside: I am inside
Outside, can see outside: I am outside
Error: name 'inside_var' is not defined


In [56]:
def title_features(title):
    words = title.split()
    word_count = len(words)

    uppercase_words = 0
    for word in words:
        if word.isupper():
            uppercase_words += 1

    marks = 0
    for ch in title:
        if ch == "!" or ch == "?":
            marks += 1

    return word_count, uppercase_words, marks

print(title_features("WE WANT TO TALK ABOUT OUR MARRIAGE!"))
print(title_features("Eruma Saani | Tamil vs English"))

(7, 7, 1)
(6, 0, 0)


In [58]:
sample = clean_records[:5]
for i in range(len(sample)):
    t = sample[i]["title"]
    print(i, t[:60], title_features(t))

0 WE WANT TO TALK ABOUT OUR MARRIAGE (7, 7, 0)
1 The Trump Presidency: Last Week Tonight with John Oliver (HB (10, 1, 0)
2 Racist Superman | Rudy Mancuso, King Bach & Lele Pons (10, 0, 0)
3 Nickelback Lyrics: Real or Fake? (5, 0, 1)
4 I Dare You: GOING BALD!? (5, 3, 2)


In [59]:
i = 0
found = None
while i < len(clean_records):
    rec = clean_records[i]
    i += 1
    if rec["views"] == 0:
        continue
    if rec["views"] > 50_000_000:
        found = rec
        break

if found:
    print(found["title"], found["views"], found["country"])
else:
    print("No video above the threshold")

Luis Fonsi, Demi Lovato - Échame La Culpa 56843038 US


## Loops
- for loop: ___
- range() with index: ___
- while loop: ___ (when does it stop?)
- continue: ___
- break: ___

In [65]:
def classify_video(record):
    views = record["views"]
    likes = record["likes"]
    dislikes = record["dislikes"]

    # Ratings disabled -> cannot judge likes
    if record["ratings_disabled"]:
        return "Unrated"

    # Zero views
    if views == 0:
        return "Low Engagement"

    # No likes and no dislikes -> avoid division by zero
    total = likes + dislikes
    if total == 0:
        return "Low Engagement"

    like_ratio = likes / total

    if like_ratio < DIVISIVE_LIKE_RATIO:
        return "Divisive"
    else:
        if views >= VIRAL_VIEWS and like_ratio >= GOOD_LIKE_RATIO:
            return "Viral Hit"
        elif views >= 100_000:
            return "Steady Performer"
        else:
            return "Low Engagement"

In [66]:
label_counts = {}
for rec in clean_records:
    label = classify_video(rec)
    if label in label_counts:
        label_counts[label] += 1
    else:
        label_counts[label] = 1

for label in label_counts:
    print(label, "=", label_counts[label])

print("Total:", sum(label_counts.values()))

Steady Performer = 60353
Viral Hit = 36590
Low Engagement = 16231
Divisive = 2821
Unrated = 1222
Total: 117217


In [67]:
for rec in clean_records:
    if rec["title"].startswith("Luis Fonsi, Demi Lovato"):
        print(classify_video(rec), rec["views"], rec["likes"], rec["dislikes"])
        break

Steady Performer 499946 135292 3528


## Hand-trace of one record
Example: "Luis Fonsi, Demi Lovato - Échame La Culpa" (US), views = 56,843,038.
1. ratings_disabled is False, so it is not "Unrated".
2. views is not 0, so it continues.
3. likes + dislikes is not 0, so division is safe.
4. like_ratio is above 0.60, so it is not "Divisive".
5. views >= 1,000,000 and like_ratio >= 0.90, so it returns "Viral Hit".

## Classification thresholds
- Unrated: ratings are disabled.
- Low Engagement: zero views, or zero likes and dislikes, or fewer than 100,000 views.
- Divisive: like ratio below 0.60.
- Viral Hit: 1,000,000+ views and like ratio 0.90 or more.
- Steady Performer: 100,000+ views (everything else that passes).